Verifying the graph database in Neo4j for the BART system
Fundamentals of Data Engineering

# Modules and Packages

In [2]:
import neo4j

import csv

import math
import numpy as np
import pandas as pd

import psycopg2

# Code

In [3]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [4]:
session = driver.session(database="neo4j")

In [5]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [6]:
def my_neo4j_shortest_path(from_station, to_station):
    "given a from station and to station, run and print the shortest path"
    
    query = "CALL gds.graph.drop('ds_graph', false)"
    session.run(query)

    query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
    session.run(query)

    query = """

    MATCH (source:Station {name: $source}), (target:Station {name: $target})
    CALL gds.shortestPath.dijkstra.stream(
        'ds_graph', 
        { sourceNode: source, 
          targetNode: target, 
          relationshipWeightProperty: 'weight'
        }
    )
    YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
    RETURN
        gds.util.asNode(sourceNode).name AS from,
        gds.util.asNode(targetNode).name AS to,
        totalCost,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
        costs
    ORDER BY index

    """

    result = session.run(query, source=from_station, target=to_station)
    
    for r in result:
        
        total_cost = int(r['totalCost'])
        
        print("\n--------------------------------")
        print("   Total Cost: ", total_cost)
        print("   Minutes: ", round(total_cost / 60.0,1))
        print("--------------------------------")
        
        nodes = r['nodes']
        costs = r['costs']
        
        i = 0
        previous = 0
        
        for n in nodes:
            
            print(n + ", " + str(int(costs[i]) - previous)  + ", " + str(int(costs[i])))
            
            previous = int(costs[i])
            i += 1
    

# Closeness Centrality

In [11]:
    
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

query = """

CALL gds.beta.closeness.stream('ds_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as closeness
ORDER BY score DESC

"""

my_neo4j_run_query_pandas(query)

,name,closeness
0,12th Street,0.147147
1,Lake Merritt,0.145833
2,West Oakland,0.145401
3,19th Street,0.141210
4,Fruitvale,0.137640
5,Embarcadero,0.136490
6,MacArthur,0.134986
7,Coliseum,0.129630
8,Montgomery Street,0.127937
9,Rockridge,0.124682


# Closeness Centrality
## Wasserman and Faust

In [10]:
    
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

query = """

CALL gds.beta.closeness.stream('ds_graph',
                               {useWassermanFaust: true}
                              )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as closeness
ORDER BY score DESC

"""

my_neo4j_run_query_pandas(query)

,name,closeness
0,12th Street,0.147147
1,Lake Merritt,0.145833
2,West Oakland,0.145401
3,19th Street,0.141210
4,Fruitvale,0.137640
5,Embarcadero,0.136490
6,MacArthur,0.134986
7,Coliseum,0.129630
8,Montgomery Street,0.127937
9,Rockridge,0.124682


# Closeness Centrality
## Harmonic Centrality

In [9]:
    
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

query = """

CALL gds.alpha.closeness.harmonic.stream('ds_graph', {})
YIELD nodeId, centrality
RETURN gds.util.asNode(nodeId).name AS name, centrality as closeness
ORDER BY centrality DESC

"""

my_neo4j_run_query_pandas(query)

,name,closeness
0,12th Street,0.233302
1,Lake Merritt,0.231867
2,West Oakland,0.228991
3,MacArthur,0.226052
4,19th Street,0.219525
5,Coliseum,0.216537
6,Fruitvale,0.214852
7,Bay Fair,0.209259
8,Embarcadero,0.204956
9,San Leandro,0.204085


# Betweenness Centrality

In [8]:
    
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

query = """

CALL gds.betweenness.stream('ds_graph', {})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as betweenness
ORDER BY betweenness DESC

"""

my_neo4j_run_query_pandas(query)

,name,betweenness
0,MacArthur,1176.0
1,12th Street,1116.0
2,19th Street,1088.0
3,Lake Merritt,1020.0
4,Fruitvale,980.0
5,West Oakland,980.0
6,Coliseum,960.0
7,Embarcadero,936.0
8,Montgomery Street,888.0
9,Powell Street,836.0


# Louvain Modularity

In [7]:
    
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

query = """

CALL gds.louvain.stream('ds_graph', {includeIntermediateCommunities: true})
YIELD nodeId, communityId, intermediateCommunityIds
RETURN gds.util.asNode(nodeId).name AS name, communityId as community, intermediateCommunityIds as intermediate_community
ORDER BY community, name ASC

"""

my_neo4j_run_query_pandas(query)

,name,community,intermediate_community
0,Ashby,2,"[15, 2]"
1,Downtown Berkeley,2,"[15, 2]"
2,El Cerrito Plaza,2,"[38, 2]"
3,El Cerrito del Norte,2,"[38, 2]"
4,North Berkeley,2,"[15, 2]"
5,Richmond,2,"[38, 2]"
6,16th Street Mission,10,"[3, 10]"
7,24th Street Mission,10,"[3, 10]"
8,Civic Center,10,"[10, 10]"
9,Glen Park,10,"[3, 10]"
